# DisasterLens on a Colab GPU

This notebook pulls the repository into the Colab runtime, mounts the official BRIGHT copy in Drive, then runs the real-data M1 audit and M2 eight-tile overfit gate. Each phase streams its progress, and generated reports, manifests, checkpoints, and metrics are copied back to Drive. It never creates a substitute dataset.

In [ ]:
import os
import select
import shutil
import time
import subprocess
import base64
from getpass import getpass
from pathlib import Path

github_token = os.environ.get("GITHUB_TOKEN")
if not github_token:
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None
if not github_token:
    github_token = getpass("GitHub token (input is hidden): ")
github_token = github_token.strip()
if not github_token:
    raise RuntimeError("No GitHub token was supplied.")

REPO_URL = os.environ.get("DISASTERLENS_REPO_URL", "https://github.com/kushc2004/disaster-lens.git")
REPO_DIR = Path("/content/disaster-lens")
git_env = os.environ.copy()
basic_auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic_auth}"})

if REPO_DIR.exists():
    if (REPO_DIR / ".git").exists():
        command = ["git", "-C", str(REPO_DIR), "pull"]
    else:
        import shutil
        shutil.rmtree(REPO_DIR)
        command = ["git", "clone", REPO_URL, str(REPO_DIR)]
else:
    command = ["git", "clone", REPO_URL, str(REPO_DIR)]
print("[setup] Syncing the repository from GitHub...", flush=True)
subprocess.run(command, env=git_env, check=True)
print(f"[setup] Repository ready at {REPO_DIR}", flush=True)
%cd /content/disaster-lens

In [ ]:
%pip install -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Select a Colab GPU kernel before running this notebook."

In [ ]:
# M1: stage the official BRIGHT data from Drive into Colab's local runtime.
from google.colab import drive
from pathlib import Path
import os
import shutil
import time

drive.mount('/content/drive')
DRIVE_BRIGHT_ROOT = Path('/content/drive/MyDrive/disaster-lens/data/raw/bright')
LOCAL_BRIGHT_ROOT = Path('/content/bright')
required = [DRIVE_BRIGHT_ROOT / name for name in ('pre-event', 'post-event', 'target')]
if not all(path.is_dir() for path in required):
    raise RuntimeError(f'Expected extracted official BRIGHT directories at {DRIVE_BRIGHT_ROOT}; found {[str(p) for p in required if not p.is_dir()]}')
LOCAL_BRIGHT_ROOT.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage('/content').free / (1024 ** 3)
print(f'[stage] Colab local free disk before copy: {free_gb:.1f} GiB', flush=True)
os.environ['DISASTERLENS_BRIGHT_ROOT'] = str(LOCAL_BRIGHT_ROOT)
ARTIFACT_DIR = Path('/content/drive/MyDrive/disaster-lens/artifacts')

def run_step(title, command):
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}", flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    buffer = ''
    fd = process.stdout.fileno()
    while True:
        ready, _, _ = select.select([fd], [], [], 0.5)
        if not ready:
            if process.poll() is not None:
                break
            continue
        raw = os.read(fd, 4096)
        if not raw:
            break
        chunk = raw.decode('utf-8', errors='replace')
        buffer += chunk.replace('\r', '\n')
        lines = buffer.split('\n')
        buffer = lines.pop()
        for line in lines:
            if line:
                print(line, flush=True)
    if buffer:
        print(buffer, flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
    print(f"[complete] {title} in {time.perf_counter() - started:.1f}s", flush=True)

def persist_artifacts():
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    for source, destination in ((REPO_DIR / 'outputs', ARTIFACT_DIR / 'outputs'), (REPO_DIR / 'data' / 'manifests', ARTIFACT_DIR / 'manifests')):
        if source.exists():
            shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f"[artifacts] persisted reports, manifests, checkpoints, and metrics to {ARTIFACT_DIR}", flush=True)

print(f"[M1] official BRIGHT source: {DRIVE_BRIGHT_ROOT}", flush=True)
for modality in ('pre-event', 'post-event', 'target'):
    destination = LOCAL_BRIGHT_ROOT / modality
    destination.mkdir(parents=True, exist_ok=True)
    run_step(f'[M1 1/4] Stage official {modality} to Colab local disk', [
        'rsync', '-a', '--partial', '--info=progress2', '--human-readable',
        f'{DRIVE_BRIGHT_ROOT}/{modality}/', f'{destination}/'])
print(f'[M1] running against staged official data at {LOCAL_BRIGHT_ROOT}', flush=True)
run_step('[M1 2/4] Audit data, alignment, labels, and normalization', ['python', '-u', 'scripts/inspect_bright.py', 'data=bright'])
run_step('[M1 3/4] Read a real DataLoader batch using the audited manifest', ['python', '-u', 'scripts/verify_bright_loader.py', 'data=bright'])
persist_artifacts()

In [ ]:
# Select one real event from outputs/reports/bright_data_audit.md, then run the M2 gate.
# Do not use a made-up event ID. The value below is deliberately required.
TEST_EVENT = ''
if not TEST_EVENT:
    raise ValueError('Set TEST_EVENT to an event ID printed by the real M1 audit, then re-run this cell.')

run_step('[M2 1/3] Create a real event-holdout split', ['python', '-u', 'scripts/make_splits.py', 'data=bright', f'split.test_events=[{TEST_EVENT}]'])
run_step('[M2 2/3] Train the eight-tile real-data overfit gate (100 epochs)', ['python', '-u', 'scripts/train.py', 'split_path=data/manifests/splits/event_holdout.json', 'overfit_tiles=8', 'trainer.epochs=100', 'trainer.crop_size=512'])
run_step('[M2 3/3] Evaluate the best checkpoint on the held-out event', ['python', '-u', 'scripts/evaluate.py', 'checkpoint=outputs/checkpoints/early_fusion_unet/best.pt', 'split_path=data/manifests/splits/event_holdout.json', 'partition=test'])
persist_artifacts()